# 损失函数：从离线评估到自定义训练

本 Notebook 使用 `examples/hscredit_yyp.xlsx` 的真实放款数据，演示 16 种损失配套指标、业务指标、金额权重、CV、HSCredit 训练和原生框架训练，以及自定义 `BaseLoss` / `BaseMetric`。

依赖：`pip install -e ".[boost,tune,dev]"`。从仓库根目录或 `examples` 目录打开，按顺序运行全部单元格。默认只训练几十轮，参数用于演示流程，不能当作已验证的生产最优值。超参数搜索完整教程见 [tuning_workflow.ipynb](tuning_workflow.ipynb)，接口说明见 [losses-guide.md](../docs/articles/losses-guide.md)。

约定 `FPD=1` 为坏样本，`predict_proba(... )[:, 1]` 是坏样本概率。训练损失用于计算梯度；配套指标用于监控和评估；业务指标用于回答实际效果问题。

In [1]:
from pathlib import Path
import sys

repository_root = next(
    (path for path in (Path.cwd(), Path.cwd().parent) if (path / "hscredit").is_dir()),
    None,
)
if repository_root is None:
    raise FileNotFoundError("请从仓库根目录或 examples 目录启动 Notebook")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from hscredit import LightGBM
from hscredit.core.models.losses import (
    BaseLoss, BaseMetric, AmountWeightedLoss, ApprovalRateLoss,
    AsymmetricFocalLoss, BadDebtLoss, BalancedFocalLoss, CostSensitiveLoss,
    ExpectedProfitLoss, ExpectedValueLoss, FocalLoss, KSFocusedLoss,
    LiftFocusedLoss, OrdinalRankLoss, ProfitMaxLoss, RankingAUCProxyLoss,
    TopKBadCaptureLoss, WeightedBCELoss, AUCMetric, BadDebtMetric,
    ProfitMetric, TopKCaptureMetric,
)

FEATURES = ["衡枢鉴真分老客版", "近六个月非银多头机构数", "青云24"]
ROUNDS = 40

## 1. 数据与独立测试集

先固定独立测试集。后续搜索只在训练集内进行，框架早停也另外从训练集划出监控集。金额随行一起切分，避免绑定金额数组与样本错位。缺失值填充和标准化放在 Pipeline 内，每个 CV 训练折独立拟合。

In [2]:
data = pd.read_excel(repository_root / "examples" / "hscredit_yyp.xlsx")
data = data.loc[data["FPD"].isin([0, 1])].copy()
X = data[FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
y = data["FPD"].astype(int)
amount = pd.to_numeric(data["放款金额"], errors="raise")
if not np.isfinite(amount).all() or (amount < 0).any() or amount.sum() <= 0:
    raise ValueError("放款金额必须有限、非负且总和大于零")
X_train, X_test, y_train, y_test, amount_train, amount_test = train_test_split(
    X, y, amount, test_size=0.3, stratify=y, random_state=42,
)

def make_baseline():
    """每次返回新的训练流水线，预处理不会看到验证折或测试集。"""
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=500))

print(f"训练样本数：{len(y_train)}；测试样本数：{len(y_test)}；测试坏样本率：{y_test.mean():.4f}")

训练样本数：679；测试样本数：291；测试坏样本率：0.1409


## 2. 最小用法：评估已经训练好的模型

`loss.metric()` 返回同口径损失指标，越小越好。无需用这个 loss 训练，也能使用它评价任意模型的坏样本概率。这里先只使用约定主评分字段。

In [3]:
baseline = make_baseline().fit(X_train[[FEATURES[0]]], y_train)
probability = baseline.predict_proba(X_test[[FEATURES[0]]])
loss = FocalLoss(alpha=0.75, gamma=2)
metric = loss.metric()
print(f"{metric.name}：{metric.evaluate(y_test, probability):.6f}；方向：{metric.direction}")
assert np.isclose(metric(y_test, probability[:, 1]), metric.evaluate(y_test, probability))

business_metrics = [AUCMetric(), TopKCaptureMetric(0.1), BadDebtMetric(0.3), ProfitMetric(100, 1000, 0.1)]
comparison = pd.DataFrame([
    {"指标": item.name, "数值": item.evaluate(y_test, probability), "优化方向": item.direction}
    for item in business_metrics
])
print(comparison.to_string(index=False))

聚焦损失：0.146113；方向：minimize
      指标      数值     优化方向
AUC曲线下面积  0.6494 maximize
头部坏样本捕获率  0.2195 maximize
 通过客户坏账率  0.0682 minimize
  人均审批利润 12.7148 maximize


## 3. 每种损失都可以获得配套指标

同一行的“损失值”与“业务指标值”不一定相等：例如排序代理损失配套的业务指标是 AUC，利润代理对应真实审批利润。表格用于展示各自口径，**不能按不同损失的数值排序比较算法优劣**。

In [4]:
losses = [
    FocalLoss(alpha=0.75), AsymmetricFocalLoss(alpha=0.75), BalancedFocalLoss(),
    WeightedBCELoss(pos_weight=3), CostSensitiveLoss(fn_cost=10), BadDebtLoss(),
    ApprovalRateLoss(), ProfitMaxLoss(), ExpectedProfitLoss(), OrdinalRankLoss(),
    LiftFocusedLoss(), RankingAUCProxyLoss(), KSFocusedLoss(), TopKBadCaptureLoss(),
    AmountWeightedLoss(amounts=amount_test.to_numpy()), ExpectedValueLoss(),
]
rows = []
for candidate in losses:
    same_loss, business = candidate.metric(), candidate.business_metric()
    rows.append({
        "损失类型": type(candidate).__name__, "同口径指标": same_loss.name,
        "损失值": same_loss.evaluate(y_test, probability), "业务指标": business.name,
        "业务指标值": business.evaluate(y_test, probability), "业务优化方向": business.direction,
    })
loss_table = pd.DataFrame(rows)
assert len(loss_table) == 16 and np.isfinite(loss_table[["损失值", "业务指标值"]].to_numpy()).all()
print(loss_table.to_string(index=False))

               损失类型    同口径指标    损失值       业务指标   业务指标值   业务优化方向
          FocalLoss     聚焦损失 0.1461       聚焦损失  0.1461 minimize
AsymmetricFocalLoss  非对称聚焦损失 0.1500    非对称聚焦损失  0.1500 minimize
  BalancedFocalLoss   平衡聚焦损失 0.1644     平衡聚焦损失  0.1644 minimize
    WeightedBCELoss    加权交叉熵 0.9232      加权交叉熵  0.9232 minimize
  CostSensitiveLoss   成本敏感损失 2.7893     人均误判成本  1.4089 minimize
        BadDebtLoss   坏账代理损失 0.4595    通过客户坏账率  0.0682 minimize
   ApprovalRateLoss  通过率代理损失 1.0945 坏账约束下最大通过率  0.2955 maximize
      ProfitMaxLoss   利润代理损失 2.7893     人均审批利润 -0.5498 maximize
 ExpectedProfitLoss   期望利润损失 0.5565     人均审批利润 -0.5498 maximize
    OrdinalRankLoss   序数排序损失 1.0696   AUC曲线下面积  0.6494 maximize
    LiftFocusedLoss   头部提升损失 0.4619     头部提升倍数  2.1293 maximize
RankingAUCProxyLoss 排序面积代理损失 0.4257   AUC曲线下面积  0.6494 maximize
      KSFocusedLoss   分布分离损失 1.0465      KS区分度  0.2468 maximize
 TopKBadCaptureLoss   头部捕获损失 0.7021   头部坏样本捕获率  0.1220 maximize
 AmountWeightedLoss   金额加权损失 0.4079     

## 4. 金额数组与样本权重

绑定金额的损失需要为训练集和评估集分别传入对应数组。`metric(amounts=...)` 只改指标快照，不修改训练 loss。普通 BCE 也可通过 `sample_weight` 表达金额加权；在 CV 中优先使用可按折切分的外部权重，不把全量金额绑进 loss。

In [5]:
train_amount_loss = AmountWeightedLoss(amounts=amount_train.to_numpy())
test_amount_metric = train_amount_loss.metric(amounts=amount_test.to_numpy())
print(f"测试集金额加权损失：{test_amount_metric.evaluate(y_test, probability):.6f}")
weighted_bce = WeightedBCELoss().evaluate(y_test, probability, sample_weight=amount_test)
print(f"按放款金额加权的交叉熵：{weighted_bce:.6f}")
assert len(train_amount_loss.amounts_) == len(y_train)

测试集金额加权损失：0.407857
按放款金额加权的交叉熵：0.407890


## 5. sklearn CV 与 GridSearchCV

sklearn scorer 统一越大越好，所以 `metric.to_scorer()` 自动把损失取负。离线 `metric.evaluate` 和 HSCredit `ModelTuner` 保留原始损失。为所有候选固定同一指标参数，不能一边改变损失口径一边比较损失大小。

In [6]:
metric = FocalLoss(alpha=0.75).metric()
search = GridSearchCV(
    make_baseline(), {"logisticregression__C": [0.1, 1.0, 10.0]},
    scoring={"聚焦损失": metric.to_scorer(), "区分度": AUCMetric().to_scorer()},
    refit="聚焦损失", cv=StratifiedKFold(3, shuffle=True, random_state=42), n_jobs=1,
).fit(X_train, y_train)
p_test = search.predict_proba(X_test)
print(f"最优正则化参数：{search.best_params_}")
print(f"交叉验证平均损失：{-search.best_score_:.6f}")
print(f"独立测试损失：{metric.evaluate(y_test, p_test):.6f}")
print(f"独立测试 AUC：{AUCMetric().evaluate(y_test, p_test):.6f}")

最优正则化参数：{'logisticregression__C': 0.1}
交叉验证平均损失：0.146388
独立测试损失：0.147473
独立测试 AUC：0.627415


## 6. HSCredit 模型直接接受 loss 和 metric

这里用训练集内部的监控集早停。`objective=loss` 且未指定 `eval_metric` 时，HSCredit LightGBM 自动配套 `loss.metric()`。显式传入 `eval_metric=loss.business_metric()` 则按业务口径监控。原始分数与概率转换由包装器处理。

In [7]:
X_fit, X_monitor, y_fit, y_monitor = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=7,
)
loss = FocalLoss(alpha=0.75)
model = LightGBM(
    objective=loss, n_estimators=ROUNDS, num_leaves=7, learning_rate=0.05,
    early_stopping_rounds=10, n_jobs=1, random_state=42,
)
model.fit(X_fit, y_fit, eval_set=[(X_monitor, y_monitor)])
print(f"测试损失：{loss.evaluate(y_test, model.predict_proba(X_test)):.6f}")
print(f"测试 AUC：{AUCMetric().evaluate(y_test, model.predict_proba(X_test)):.6f}")
assert model.predict_proba(X_test).shape == (len(X_test), 2)

测试损失：0.051436
测试 AUC：0.555756


## 7. 自定义指标与自定义损失

仅改变评价标准：继承 `BaseMetric`，实现接收标签与坏样本概率的 `__call__`，声明方向。这样同一个对象可离线评估、训练监控或传给 `ModelTuner(metric=...)`。

In [8]:
class BrierMetric(BaseMetric):
    """均方概率误差：越小表示概率与观察标签更接近。"""
    def __init__(self):
        super().__init__(name="均方概率误差", greater_is_better=False)

    def __call__(self, y_true, y_pred, sample_weight=None):
        error = (np.asarray(y_pred) - np.asarray(y_true)) ** 2
        return float(np.average(error, weights=sample_weight))

brier_metric = BrierMetric()
print(f"Brier 指标：{brier_metric.evaluate(y_test, p_test):.6f}；方向：{brier_metric.direction}")
print(f"金额加权 Brier：{brier_metric.evaluate(y_test, p_test, sample_weight=amount_test):.6f}")

Brier 指标：0.118055；方向：minimize
金额加权 Brier：0.125268


想让训练本身优化新目标：继承 `BaseLoss`。下例对**概率**求导，框架适配器负责 sigmoid 链式求导。`loss_values` 的均值等于损失；梯度/二阶导数采用“样本数 × 平均损失”的标度。可逐样本相加时声明 `is_additive=True`，才能安全使用加权和分批训练。

In [9]:
class BrierLoss(BaseLoss):
    """均方概率误差训练目标，自动获得同口径 LossMetric。"""
    is_additive = True

    def __init__(self):
        super().__init__("均方概率损失")

    def loss_values(self, y_true, y_pred):
        return (np.asarray(y_pred) - np.asarray(y_true)) ** 2

    def __call__(self, y_true, y_pred):
        return float(np.mean(self.loss_values(y_true, y_pred)))

    def gradient(self, y_true, y_pred):
        return 2 * (np.asarray(y_pred) - np.asarray(y_true))

    def hessian(self, y_true, y_pred):
        return np.full(len(y_true), 2.0)

custom_loss = BrierLoss()
custom_model = LightGBM(objective=custom_loss, n_estimators=ROUNDS, num_leaves=7, n_jobs=1)
custom_model.fit(X_fit, y_fit, eval_set=[(X_monitor, y_monitor)])
custom_probability = custom_model.predict_proba(X_test)
assert np.isclose(custom_loss.evaluate(y_test, custom_probability), brier_metric.evaluate(y_test, custom_probability))
print(f"自定义目标模型测试损失：{custom_loss.evaluate(y_test, custom_probability):.6f}")

自定义目标模型测试损失：0.121532


## 8. 原生 XGBoost 与 LightGBM

如果已有原生训练流程，分别转换 objective 和 metric。自定义目标回调接收原始分数，配套 metric 必须声明 `raw_score=True`。XGBoost 原生 `train` 还需 `maximize`；LightGBM metric 回调自带方向。

In [10]:
import xgboost as xgb

imputer = SimpleImputer(strategy="median").fit(X_fit)
fit_values, monitor_values, test_values = [imputer.transform(part) for part in (X_fit, X_monitor, X_test)]
loss, metric = FocalLoss(alpha=0.75), FocalLoss(alpha=0.75).metric()
train_data = xgb.DMatrix(fit_values, label=y_fit)
monitor_data = xgb.DMatrix(monitor_values, label=y_monitor)
native_xgb = xgb.train(
    {"max_depth": 3, "eta": 0.05, "disable_default_eval_metric": 1, "nthread": 1, "seed": 42},
    train_data, num_boost_round=ROUNDS, obj=loss.to_xgboost(),
    custom_metric=metric.to_xgboost(raw_score=True), evals=[(monitor_data, "监控集")],
    maximize=metric.greater_is_better, early_stopping_rounds=10, verbose_eval=False,
)
margin = native_xgb.predict(
    xgb.DMatrix(test_values), output_margin=True, iteration_range=(0, native_xgb.best_iteration + 1),
)
print(f"XGBoost 测试损失：{metric.evaluate(y_test, expit(margin)):.6f}")

XGBoost 测试损失：0.051036


In [11]:
import lightgbm as lgb

train_data = lgb.Dataset(fit_values, label=y_fit)
monitor_data = lgb.Dataset(monitor_values, label=y_monitor, reference=train_data)
native_lgb = lgb.train(
    {"objective": loss.to_lightgbm(api="native"), "metric": "None", "learning_rate": 0.05,
     "num_leaves": 7, "num_threads": 1, "verbosity": -1, "seed": 42},
    train_data, num_boost_round=ROUNDS, valid_sets=[monitor_data], valid_names=["监控集"],
    feval=metric.to_lightgbm(api="native", raw_score=True), callbacks=[lgb.early_stopping(10, verbose=False)],
)
margin = native_lgb.predict(test_values, raw_score=True, num_iteration=native_lgb.best_iteration)
print(f"LightGBM 测试损失：{metric.evaluate(y_test, margin, raw_score=True):.6f}")
assert np.isclose(metric.evaluate(y_test, margin, raw_score=True), metric.evaluate(y_test, expit(margin)))

LightGBM 测试损失：0.051436


## 9. 下一步与口径检查

- 需要搜索模型或 loss 参数：继续 [超参数搜索 Notebook](tuning_workflow.ipynb)，固定评估口径后再搜索训练目标参数。
- 需要早停：使用训练内部监控集；保留独立测试集用于最终比较。
- 需要金额或其他逐样本业务数组：与当前数据按同一行位置切分，不把训练数组复用于其他样本。
- `loss.metric()` 衡量训练代理，`loss.business_metric()` 衡量对应业务结果；二者都不能保证未来分布上的表现。
- CatBoost、NGBoost、TabNet 的分批目标限制、原生回调和全部损失解释见 [完整损失指南](../docs/articles/losses-guide.md)。